In [1]:
# CELL 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH    = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/Dataset/'
RESULTS_PATH = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_ Comparator Baselines/Results/LambdaMART_Bagging/'

import os
os.makedirs(RESULTS_PATH, exist_ok=True)
print('Drive mounted!')

Mounted at /content/drive
Drive mounted!


In [2]:
# CELL 2: Imports
!pip install lightgbm scikit-learn numpy pandas scipy -q

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from scipy import stats
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
print('Imports done!')

Imports done!


In [3]:
# CELL 3: Schemes and file paths
NORM_COLS = [
    'Response_Time_norm',
    'Availability_norm',
    'Throughput_norm',
    'Reliability_norm',
    'Latency_norm'
]

SCHEMES = {
    'A': {
        'description': 'Equal weights',
        'train_file': 'Scheme_A/obj3_train_A.csv',
        'val_file':   'Scheme_A/obj3_val_A.csv',
        'test_file':  'Scheme_A/obj3_test_A.csv'
    },
    'B': {
        'description': 'Latency-dominant',
        'train_file': 'Scheme_B/obj3_train_B.csv',
        'val_file':   'Scheme_B/obj3_val_B.csv',
        'test_file':  'Scheme_B/obj3_test_B.csv'
    },
    'C': {
        'description': 'Availability-dominant',
        'train_file': 'Scheme_C/obj3_train_C.csv',
        'val_file':   'Scheme_C/obj3_val_C.csv',
        'test_file':  'Scheme_C/obj3_test_C.csv'
    },
    'D': {
        'description': 'Throughput-dominant',
        'train_file': 'Scheme_D/obj3_train_D.csv',
        'val_file':   'Scheme_D/obj3_val_D.csv',
        'test_file':  'Scheme_D/obj3_test_D.csv'
    }
}
print('Schemes defined!')

Schemes defined!


In [4]:
# CELL 4: Evaluation metrics + binary Top-1 vector

def compute_mae(true_ranks, pred_ranks):
    return np.mean(np.abs(np.array(true_ranks) - np.array(pred_ranks)))

def compute_top1(true_ranks, pred_ranks):
    return int(np.argmin(pred_ranks) == np.argmin(true_ranks))

def compute_spearman(true_ranks, pred_ranks):
    corr, _ = spearmanr(true_ranks, pred_ranks)
    return corr if not np.isnan(corr) else 0.0

def compute_ndcg(true_ranks, pred_ranks, k=10):
    n = len(true_ranks)
    true_ranks = np.array(true_ranks)
    pred_ranks = np.array(pred_ranks)
    relevance  = (n + 1) - true_ranks
    pred_order = np.argsort(pred_ranks)
    sorted_rel = relevance[pred_order]
    dcg  = sum(sorted_rel[i] / np.log2(i + 2) for i in range(min(k, n)))
    idcg = sum(np.sort(relevance)[::-1][i] / np.log2(i + 2) for i in range(min(k, n)))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_method(test_df, pred_col):
    mae_l, top1_l, sp_l, ndcg_l = [], [], [], []
    for lid in test_df['list_id'].unique():
        g = test_df[test_df['list_id'] == lid]
        tr = g['list_rank'].values
        pr = g[pred_col].values
        mae_l.append(compute_mae(tr, pr))
        top1_l.append(compute_top1(tr, pr))
        sp_l.append(compute_spearman(tr, pr))
        ndcg_l.append(compute_ndcg(tr, pr))
    metrics = {
        'MAE':      round(np.mean(mae_l), 4),
        'Top-1':    round(np.mean(top1_l) * 100, 2),
        'Spearman': round(np.mean(sp_l), 4),
        'NDCG':     round(np.mean(ndcg_l), 4)
    }
    return metrics, np.array(top1_l), np.array(mae_l), np.array(sp_l), np.array(ndcg_l)

In [5]:
# CELL 5: LambdaMART with bagging and feature fraction

def run_lambdamart_bagging(train_df, val_df, test_df, seed=42):
    def prep(df):
        X, y, g = [], [], []
        for lid in df['list_id'].unique():
            grp = df[df['list_id'] == lid]
            X.append(grp[NORM_COLS].values)
            y.append((11 - grp['list_rank']).values)
            g.append(len(grp))
        return np.vstack(X), np.concatenate(y), g

    X_tr, y_tr, g_tr = prep(train_df)
    X_vl, y_vl, g_vl = prep(val_df)
    X_te, _,    _    = prep(test_df)

    dtrain = lgb.Dataset(X_tr, label=y_tr, group=g_tr)
    dval   = lgb.Dataset(X_vl, label=y_vl, group=g_vl)

    params = {
        'objective':        'lambdarank',
        'metric':           'ndcg',
        'ndcg_eval_at':     [1, 5, 10],
        'learning_rate':    0.05,
        'num_leaves':       31,
        'verbose':          -1,
        'seed':             seed,
        'bagging_fraction': 0.9,
        'bagging_freq':     5,
        'feature_fraction': 0.9,
    }

    model = lgb.train(
        params, dtrain,
        num_boost_round=150,
        valid_sets=[dval],
        callbacks=[lgb.log_evaluation(50)]
    )

    scores = model.predict(X_te)
    test_df = test_df.copy()
    test_df['lgb_score'] = scores
    ranks = []
    for lid in test_df['list_id'].unique():
        g = test_df[test_df['list_id'] == lid]
        r = g['lgb_score'].rank(ascending=False, method='min').astype(int)
        ranks.extend(r.values)
    test_df['lgb_rank'] = ranks
    return evaluate_method(test_df, 'lgb_rank')

In [6]:
# CELL 6: Run LambdaMART with bagging across 5 seeds and all schemes

def aggregate_runs(runs_metrics, runs_top1_vectors):
    result = {}
    for metric in runs_metrics[0]:
        scores = [r[metric] for r in runs_metrics]
        mean = np.mean(scores)
        sem  = stats.sem(scores)
        if np.isnan(sem) or sem == 0:
            margin = 0.0
        else:
            ci = stats.t.interval(0.95, len(scores)-1, loc=mean, scale=sem)
            margin = round(mean - ci[0], 4)
        result[metric]         = round(mean, 4)
        result[f'{metric}_ci'] = margin
    return result, runs_top1_vectors[0]

seeds = [42, 123, 456, 789, 1011]
all_results   = {}
all_top1_vecs = {}

for scheme, info in SCHEMES.items():
    print('='*60)
    print(f'SCHEME {scheme}: {info["description"]}')
    print('='*60)

    train_df = pd.read_csv(BASE_PATH + info['train_file'])
    val_df   = pd.read_csv(BASE_PATH + info['val_file'])
    test_df  = pd.read_csv(BASE_PATH + info['test_file'])

    lgb_metrics_runs = []
    lgb_top1_runs    = []
    lgb_mae_runs     = []
    lgb_sp_runs      = []
    lgb_ndcg_runs    = []

    for s in seeds:
        print(f'  Seed {s}...')
        m, v_top1, v_mae, v_sp, v_ndcg = run_lambdamart_bagging(train_df, val_df, test_df, seed=s)
        lgb_metrics_runs.append(m)
        lgb_top1_runs.append(v_top1)
        lgb_mae_runs.append(v_mae)
        lgb_sp_runs.append(v_sp)
        lgb_ndcg_runs.append(v_ndcg)
        print(f'    MAE={m["MAE"]}, Top-1={m["Top-1"]}')

    agg_m, agg_v = aggregate_runs(lgb_metrics_runs, lgb_top1_runs)
    all_results[f'Scheme_{scheme}']   = agg_m
    all_top1_vecs[f'Scheme_{scheme}'] = agg_v
    print(f'  Aggregated: {agg_m}')

    # Save aggregate results
    pd.DataFrame([agg_m]).to_csv(
        RESULTS_PATH + f'lambdamart_bagging_results_scheme_{scheme}.csv', index=False)

    # Save per-list vectors for seed 42 (for paired t-tests)
    seed42_idx = seeds.index(42)
    pd.DataFrame({
        'LambdaMART_bagging': lgb_top1_runs[seed42_idx],
        'mae':                lgb_mae_runs[seed42_idx],
        'spearman':           lgb_sp_runs[seed42_idx],
        'ndcg':               lgb_ndcg_runs[seed42_idx]
    }).to_csv(RESULTS_PATH + f'lambdamart_bagging_top1_vectors_scheme_{scheme}.csv', index=False)

    print(f'  Saved to: {RESULTS_PATH}')

SCHEME A: Equal weights
  Seed 42...
[50]	valid_0's ndcg@1: 0.970153	valid_0's ndcg@5: 0.982716	valid_0's ndcg@10: 0.986323
[100]	valid_0's ndcg@1: 0.97115	valid_0's ndcg@5: 0.985634	valid_0's ndcg@10: 0.988422
[150]	valid_0's ndcg@1: 0.973081	valid_0's ndcg@5: 0.98725	valid_0's ndcg@10: 0.989379
    MAE=0.2526, Top-1=95.22
  Seed 123...
[50]	valid_0's ndcg@1: 0.969093	valid_0's ndcg@5: 0.983404	valid_0's ndcg@10: 0.98678
[100]	valid_0's ndcg@1: 0.975075	valid_0's ndcg@5: 0.986369	valid_0's ndcg@10: 0.988917
[150]	valid_0's ndcg@1: 0.977069	valid_0's ndcg@5: 0.988127	valid_0's ndcg@10: 0.990244
    MAE=0.2629, Top-1=92.43
  Seed 456...
[50]	valid_0's ndcg@1: 0.97115	valid_0's ndcg@5: 0.98348	valid_0's ndcg@10: 0.986838
[100]	valid_0's ndcg@1: 0.969093	valid_0's ndcg@5: 0.984797	valid_0's ndcg@10: 0.987381
[150]	valid_0's ndcg@1: 0.973081	valid_0's ndcg@5: 0.987281	valid_0's ndcg@10: 0.98928
    MAE=0.2566, Top-1=94.42
  Seed 789...
[50]	valid_0's ndcg@1: 0.962114	valid_0's ndcg@5: 0.98

In [7]:
# CELL 7: Summary

print('LAMBDAMART BAGGING RESULTS SUMMARY')
print('='*60)
for scheme_key, metrics in all_results.items():
    print(f'\n{scheme_key}:')
    for metric, value in metrics.items():
        if '_ci' not in metric:
            ci = metrics.get(f'{metric}_ci', 0)
            print(f'  {metric}: {value} ± {ci}')

LAMBDAMART BAGGING RESULTS SUMMARY

Scheme_A:
  MAE: 0.2596 ± 0.006
  Top-1: 93.626 ± 1.5248
  Spearman: 0.9766 ± 0.0007
  NDCG: 0.9968 ± 0.0002

Scheme_B:
  MAE: 0.2977 ± 0.0099
  Top-1: 92.99 ± 1.0299
  Spearman: 0.9713 ± 0.0013
  NDCG: 0.9964 ± 0.0001

Scheme_C:
  MAE: 0.1795 ± 0.0037
  Top-1: 96.254 ± 0.8252
  Spearman: 0.9866 ± 0.0004
  NDCG: 0.9983 ± 0.0002

Scheme_D:
  MAE: 0.1614 ± 0.0044
  Top-1: 95.14 ± 1.5389
  Spearman: 0.988 ± 0.0005
  NDCG: 0.9984 ± 0.0002
